# Save pre-processed data per exercise

In [ ]:
from preprocessing import audio_processor
from data import humv_loader
import os
import numpy as np
import pandas as pd

# Define root directory and output folder
root_directory = '/home/marcos/Documentos/GitHub/TFM_code/data/raw'
output_folder = '/home/marcos/Documentos/GitHub/TFM_code/data/processed'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    ("2 audios of 5 seconds", ['aueoi.wav', 'ka.wav', 'pa.wav', 'patachaka.wav', 'pataka.wav', 'ta.wav', 'uy.wav', 'vocal.wav'], 0, 5, 10),
    ("6 audios of 5 seconds", ['habla libre.wav', 'robo galletas.wav'], 0, 5, 35),
    ("10 audios of 5 seconds", ['fluencia categorial.wav'], 0, 5, 60),
    ("25 audios of 5 seconds", ['lectura texto.wav'], 0, 5, 150),
]

for desc, exact_names_values, start_time, chunk_duration, max_duration in exercise_sets:
    
    print(f"\nProcessing set: {desc}")
    
    for exact_names in exact_names_values:
        print(f"  Processing audio files: {exact_names}")
        df = humv_loader.load_audio_data(root_directory, start_with=None, exact_name=exact_names)
        df = df[df['Label'].isin([0, 2])]
        df['Label'] = df['Label'].replace(2, 1)
        
        # List of patient IDs to remove
        patients_to_remove = ['HUMV_HC_4', 'HUMV_HC_9', 'HUMV_HC_25', 'HUMV_NFC_1', 'HUMV_NFC_11', 'HUMV_NFC_24' ,'HUMV_PD_1', 'HUMV_PD_2', 'HUMV_PD_9', 'HUMV_PD_13']
        
        print("Removing patients:", patients_to_remove)
        
        # Remove duplicates just in case
        patients_to_remove = list(set(patients_to_remove))
        
        # Filter the dataframe
        df_filtered = df[~df['Patient'].isin(patients_to_remove)].copy()

        value_counts = df_filtered['Label'].value_counts()
        print(value_counts) 

        audio_segments, labels_np, patient_ids, exercises = audio_preprocessing.execute_preprocess_and_split(
            df_filtered,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration,
            target_sr=16000,
            remove_silence=True,
            top_db=20,
            silence_duration=2,
            file_path_column='File_Path',
            overlap=1,
            min_chunk_length=0.8,
        )
        file_name_without_extension = os.path.splitext(exact_names)[0]
        np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{file_name_without_extension}.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{file_name_without_extension}.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{file_name_without_extension}.npy"), patient_ids)
        np.save(os.path.join(output_folder, f"exercises_5s_with_1s_overlap_{file_name_without_extension}.npy"), exercises)
        print(f"Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")